Put together the full data set.

- remove MINFLUX U2OS, MEF, RPE stuck fraction and save separately. We do this by tagging the stuck fraction and saving a selection of the full data set using `nl.io.write.hdf5_subTaggedSet`. So the stuck fraction can be loaded from the same file, simply by using `data_with_stuck` instead of `data`.

In [1]:
import numpy as np
import noctiluca as nl

/home/sgh/gitlibs/chromatin_dynamics/.venv_SD_py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load existing data set

In [47]:
filename = '/data/sgh/science/2024_minflux/20250411_chromatin_dynamics_all_data.h5'
tmp = nl.io.load.hdf5(filename)

H2B_conditions_existing = tmp['H2B_conditions']
array_conditions_existing = tmp['array_conditions']
data_existing_with_stuck = tmp['data_withU2OSstuck']
print(tmp['description'])

Joint file containing all our chromatin dynamics data

The production dataset is stored in 'data'; 'data_withU2OSstuck' is a version of
that which also contains the "stuck" trajectories of minflux U2OS (labelled as such)

Tags:
 - 'U2OS', 'mESC'
 - 'H2B', 'array'
 - 'minflux', 'SPT', 'SRLCI'
 - 'ctrl', 'DMSO', 'DRB', 'TSA', 'ICRF' ('ctrl' and 'DMSO' are identical)
 - 'minflux-H2B', 'minflux-array'
 - 'SPT-100ms', 'SPT-2s', '100ms', '2s'
 - all combinations: "H2B, {'U2OS', 'mESC'}, {'ctrl', 'DRB', 'TSA', 'ICRF'}"
 - 'C36', 'ΔRAD21 (inactive)', 'ΔRAD21 (active)'
 - 'rep1', 'rep2', 'rep3' (except SRLCI data)
 - 'file=<original filename>'
 - 'stuck' indicates low-α trajectories in minflux U2OS data
   (only applicable when loading 'data_withU2OSstuck')
 
Notes:
 - minflux data is true single particle tracking, in 2D.
 - SPT H2B data is two-locus, in 2D; inter-locus distance is constrained to <3μm.
   The trajectories contain the full 2-locus data, but have MSDs for relative distance pre-ca

In [48]:
H2B_conditions_existing

['H2B, U2OS, DRB',
 'H2B, U2OS, ICRF',
 'H2B, U2OS, TSA',
 'H2B, U2OS, ctrl',
 'H2B, mESC, DRB',
 'H2B, mESC, ICRF',
 'H2B, mESC, TSA',
 'H2B, mESC, ctrl']

# Load new data for revision

# MINFLUX

In [9]:
#filename = '../data/20250302_minflux_H2B_clean.h5'
filename = '/data/sgh/science/2024_minflux/MINFLUX/20251115_SanDiego_prod/20251126_clean.h5'
tmp = nl.io.load.hdf5(filename)

data_mf    = tmp['data']
conditions = tmp['conditions']
print(tmp['description'])
del tmp

Cleaned-up MINFLUX H2B data

Cleaning was done by cutting before the first and after the last median-crossing

Conditions are:
 - H2B, MEF
 - H2B, RH30
 - H2B, RPE
 - H2B, U2OS
 - H2B, mESC


In [10]:
# Add single-traj mci
# this should be done first thing, because the MCIs stored in the file
# are matched to the dataset in 20250302_minflux_H2B_clean.h5.
filename = '/data/sgh/science/2024_minflux/MINFLUX/20251115_SanDiego_prod/20251126_single-traj_NPFits.h5'
mcis = nl.io.load.hdf5(filename, '/mcis')
data_mf.makeSelection()
assert len(data_mf) == len(mcis)
for traj, mci in zip(data_mf, mcis):
    traj.meta['mci'] = mci

del mcis

In [12]:
# Adjust replicate tags for consistency
for r in range(10):
    data_mf.makeSelection([f'rep={r}', f'rep=rep{r}'], logic=any)
    data_mf.addTags(f'rep{r}')

In [26]:
# Time lag (for later)
data_mf.makeSelection()
data_existing_with_stuck.makeSelection('minflux')
dt_H2B = np.mean([traj.meta['dt'] for traj in data_mf] + [traj.meta['dt'] for traj in data_existing_with_stuck])

In [18]:
# done
data_mf.makeSelection()
data_mf.addTags({'minflux', 'H2B', 'minflux-H2B'})
data_mf.addTags({'ctrl', 'NT'})

In [49]:
conditions = [cond + ', ctrl' for cond in conditions]
conditions

['H2B, MEF, ctrl',
 'H2B, RH30, ctrl',
 'H2B, RPE, ctrl',
 'H2B, U2OS, ctrl',
 'H2B, mESC, ctrl']

In [50]:
for cond in conditions:
    data_mf.makeSelection(cond[:-6])
    assert len(data_mf) > 0
    data_mf.addTags(cond)

# SPT

In [15]:
# filename = '../data/20250121_SPT_H2B.h5'
filename = '/data/sgh/science/2024_minflux/SPT/20260105_SPT_H2B_revision.h5'
data_conv = nl.io.load.hdf5(filename, 'data_twoLocus_acyclic_3um')

In [20]:
# add useful tags
data_conv.makeSelection()
data_conv.addTags('ctrl')

for cond in conditions:
    data_conv.makeSelection(cond.split(', ')[1:], logic=all)
    data_conv.addTags(cond)

for dt_tag in ['100ms', '2s']:
    data_conv.makeSelection(dt_tag)
    data_conv.addTags(f'SPT-{dt_tag}')

In [21]:
# remove what we're not using (500nM-TSA & 1μM-TSA)
data_conv.makeSelection(conditions, logic=lambda x: not any(x))
# data_conv.deleteSelection()
len(data_conv) # check: nothing to remove here

0

In [16]:
# # remove dead cells
# def dead_cells(traj, tags):
#     dead_list = [
#         ({'100ms', 'mESC', 'ctrl', 'rep3'}, ['000cell']),
#         ({'100ms', 'mESC', 'ICRF', 'rep3'}, ['000cell']),
#         ({  '2s',  'mESC', 'ctrl', 'rep3'}, ['000cell', '001cell', '002cell', '003cell']),
#     ]
    
#     for dtags, cells in dead_list:
#         if len(tags & dtags) == len(dtags): # all tags contains all dtags
#             filename = {tag for tag in tags if tag.startswith('file=')}.pop()
#             return any(c in filename for c in cells)
#     return False

# data_conv.makeSelection(dead_cells)
# data_conv.deleteSelection()

In [22]:
# done
data_conv.makeSelection()
data_conv.addTags({'SPT', 'H2B'})

# Merge all together

In [40]:
data_existing_with_stuck.makeSelection()
data_mf.makeSelection()
data_conv.makeSelection()

data = nl.TaggedSet()
data |= data_existing_with_stuck
data |= data_mf
data |= data_conv

In [41]:
# Add time lags to everything
time_lags = {
    'minflux-H2B'   : dt_H2B,
    # 'minflux-array' : dt_array,
    'SPT-100ms'     : 0.1,
    'SPT-2s'        : 2,
    # 'SRLCI'         : 20,
}
for tag, dt in time_lags.items():
    data.makeSelection(tag)
    for traj in data:
        traj.meta['Δt'] = dt

In [42]:
# check
data.makeSelection(lambda traj, _: 'Δt' not in traj.meta)
len(data)

0

In [51]:
# Remove unnecessary tags
data.makeSelection()
useful_tags = {
    'U2OS', 'mESC', 'MEF', 'RPE', 'RH30',
    'H2B', 'array', 'minflux', 'SPT', 'SRLCI',
    'ctrl', 'DMSO', 'DRB', 'TSA', 'ICRF',
    'minflux-H2B', 'minflux-array', 'SPT-100ms', 'SPT-2s', '100ms', '2s',
    *conditions, *H2B_conditions_existing, *array_conditions_existing,
    'C36', 'ΔRAD21 (inactive)', 'ΔRAD21 (active)',
    'rep1', 'rep2', 'rep3',
} | {tag for tag in data.tagset() if tag.startswith('file=')}

for traj, tags in data(giveTags=True):
    tags &= useful_tags

In [52]:
# Add "stuck" tag to minflux U2OS, 
data.makeSelection('minflux')
data.refineSelection(['U2OS', 'MEF', 'RPE'], logic=any)
data.refineSelection(lambda traj, _: traj.meta['mci']['α (dim 0)'][1][1] < 0.1)
data.addTags('stuck')

In [54]:
# Save
# outfile = '../data/20250411_chromatin_dynamics_all_data.h5'
outfile = '/data/sgh/science/2024_minflux/20260106_chromatin_dynamics_all_data.h5'
data.makeSelection()
nl.io.write.hdf5({
    'data_with_stuck' : data,
    'H2B_conditions' : sorted({*conditions, *H2B_conditions_existing}),
    'array_conditions' : array_conditions_existing,
    'description' : """
Joint file containing all our chromatin dynamics data, including NSMB revision.

The production dataset is stored in 'data'; 'data_with_stuck' is a version of
that which also contains the "stuck" trajectories of minflux U2OS, MEF, RPE (labelled as such)

Tags:
 - 'U2OS', 'mESC', 'MEF', 'RPE', 'RH30'
 - 'H2B', 'array'
 - 'minflux', 'SPT', 'SRLCI'
 - 'ctrl', 'DMSO', 'DRB', 'TSA', 'ICRF'
 - 'minflux-H2B', 'minflux-array'
 - 'SPT-100ms', 'SPT-2s', '100ms', '2s'
 - all combinations: "H2B, {'U2OS', 'mESC'}, {'ctrl', 'DRB', 'TSA', 'ICRF'}"
 - all combinations: "H2B, {'MEF', 'RPE', 'RH30'}, ctrl"
 - 'C36', 'ΔRAD21 (inactive)', 'ΔRAD21 (active)'
 - 'rep1', 'rep2', 'rep3' (except SRLCI data)
 - 'file=<original filename>'
 - 'stuck' indicates low-α trajectories in minflux data
   (only applicable when loading 'data_with_stuck')
 
Notes:
 - minflux data is true single particle tracking, in 2D.
 - SPT H2B data is two-locus, in 2D; inter-locus distance is constrained to <3μm.
   The trajectories contain the full 2-locus data, but have MSDs for relative distance pre-calculated.
 - SPT array data is two-locus, with CTCF as reference; inter-locus distance is <3μm.
 - SRLCI data is relative position of two loci, in 3D.
"""[1:-1],
}, outfile)

data.makeSelection('stuck', logic=lambda x: not any(x))
nl.io.write.hdf5_subTaggedSet(data,
                              filename=outfile,
                              group='/data',
                              refTaggedSet='/data_with_stuck',
                             )

# Sanity checks

In [55]:
conditions = [
    ['H2B', 'minflux', 'mESC', 'ctrl'],
    ['H2B', 'minflux', 'mESC', 'DRB'],
    ['H2B', 'minflux', 'mESC', 'TSA'],
    ['H2B', 'minflux', 'mESC', 'ICRF'],
    ['H2B', 'minflux', 'U2OS', 'ctrl'],
    ['H2B', 'minflux', 'U2OS', 'DRB'],
    ['H2B', 'minflux', 'U2OS', 'TSA'],
    ['H2B', 'minflux', 'U2OS', 'ICRF'],
    ['H2B', 'minflux', 'MEF', 'ctrl'],
    ['H2B', 'minflux', 'RPE', 'ctrl'],
    ['H2B', 'minflux', 'RH30', 'ctrl'],
    ['H2B', 'SPT-100ms', 'mESC', 'ctrl'],
    ['H2B', 'SPT-100ms', 'mESC', 'DRB'],
    ['H2B', 'SPT-100ms', 'mESC', 'TSA'],
    ['H2B', 'SPT-100ms', 'mESC', 'ICRF'],
    ['H2B', 'SPT-100ms', 'U2OS', 'ctrl'],
    ['H2B', 'SPT-100ms', 'U2OS', 'DRB'],
    ['H2B', 'SPT-100ms', 'U2OS', 'TSA'],
    ['H2B', 'SPT-100ms', 'U2OS', 'ICRF'],
    ['H2B', 'SPT-100ms', 'MEF', 'ctrl'],
    ['H2B', 'SPT-100ms', 'RPE', 'ctrl'],
    ['H2B', 'SPT-100ms', 'RH30', 'ctrl'],
    ['H2B', 'SPT-2s', 'mESC', 'ctrl'],
    ['H2B', 'SPT-2s', 'mESC', 'DRB'],
    ['H2B', 'SPT-2s', 'mESC', 'TSA'],
    ['H2B', 'SPT-2s', 'mESC', 'ICRF'],
    ['H2B', 'SPT-2s', 'U2OS', 'ctrl'],
    ['H2B', 'SPT-2s', 'U2OS', 'DRB'],
    ['H2B', 'SPT-2s', 'U2OS', 'TSA'],
    ['H2B', 'SPT-2s', 'U2OS', 'ICRF'],
    ['H2B', 'SPT-2s', 'MEF', 'ctrl'],
    ['H2B', 'SPT-2s', 'RPE', 'ctrl'],
    ['H2B', 'SPT-2s', 'RH30', 'ctrl'],
    ['array', 'minflux', 'C36'],
    ['array', 'minflux', 'ΔRAD21 (inactive)'],
    ['array', 'minflux', 'ΔRAD21 (active)'],
    ['array', 'SPT-100ms', 'C36'],
    ['array', 'SPT-100ms', 'ΔRAD21 (inactive)'],
    ['array', 'SPT-100ms', 'ΔRAD21 (active)'],
    ['array', 'SPT-2s', 'C36'],
    ['array', 'SPT-2s', 'ΔRAD21 (inactive)'],
    ['array', 'SPT-2s', 'ΔRAD21 (active)'],
    ['array', 'SRLCI', 'C36'],
    ['array', 'SRLCI', 'ΔRAD21 (inactive)'],
    ['array', 'SRLCI', 'ΔRAD21 (active)'],
]

In [56]:
# Trajectories are uniquely associated with one of the above conditions
data.makeSelection()
success = True
for _, tags in data(giveTags=True):
    cnt = np.sum([all(tag in tags for tag in cond) for cond in conditions])
    if cnt != 1:
        success = False
        print(cnt, tags)

if success:
    print('All trajectories uniquely associated with one condition')

# Same check for reps within conditions
for cond in conditions:
    data.makeSelection(cond, logic=all)
    for _, tags in data(giveTags=True):
        cnt = len(tags & {'rep1', 'rep2', 'rep3'})
        if cnt != 1:
            success = False
            print(cnt, tags)

if success:
    print('All trajectories uniquely associated with one repeat within each condition')

All trajectories uniquely associated with one condition
All trajectories uniquely associated with one repeat within each condition


In [57]:
print("Trajectory count per condition and repeat")
print("-----------------------------------------")
for cond in conditions:
    data.makeSelection(cond, logic=all)
    reptags = sorted({tag for tag in data.tagset() if tag.startswith('rep')})
    
    if len(reptags) == 0:
        lens = [len(data)]
    else:
        lens = []
        for tag in reptags:
            data.makeSelection(cond, logic=all)
            data.refineSelection(tag)
            lens.append(len(data))
    
    # print(f'{str(cond):<45s}, {str(reptags):<30s}, {str(lens):>20s}')
    print(f'{str(cond):<45s}, {str(lens):>20s}')

Trajectory count per condition and repeat
-----------------------------------------
['H2B', 'minflux', 'mESC', 'ctrl']           ,      [269, 435, 300]
['H2B', 'minflux', 'mESC', 'DRB']            ,      [218, 150, 157]
['H2B', 'minflux', 'mESC', 'TSA']            ,      [159, 180, 146]
['H2B', 'minflux', 'mESC', 'ICRF']           ,      [162, 233, 190]
['H2B', 'minflux', 'U2OS', 'ctrl']           ,      [754, 696, 684]
['H2B', 'minflux', 'U2OS', 'DRB']            ,      [465, 395, 443]
['H2B', 'minflux', 'U2OS', 'TSA']            ,      [582, 305, 486]
['H2B', 'minflux', 'U2OS', 'ICRF']           ,      [487, 410, 498]
['H2B', 'minflux', 'MEF', 'ctrl']            ,      [221, 262, 276]
['H2B', 'minflux', 'RPE', 'ctrl']            ,      [130, 145, 129]
['H2B', 'minflux', 'RH30', 'ctrl']           ,        [97, 81, 162]
['H2B', 'SPT-100ms', 'mESC', 'ctrl']         ,   [1254, 1450, 1788]
['H2B', 'SPT-100ms', 'mESC', 'DRB']          ,      [541, 940, 933]
['H2B', 'SPT-100ms', 'mESC', 'TS